In [1]:
# import libraries
import pandas as pd
pd.set_option('display.max_columns', None)
import geopandas as gpd

from rapidfuzz import process, fuzz

#### Import the data

In [2]:
files = {
    'auto':'Auto_Theft_Open_Data.csv',
    'bike':'Bike_Theft_Open_Data.csv',
    'calls': 'Calls_For_Service.csv',
    'crimes': 'Criminal_Offences_Open_Data.csv',
    'homicide': 'Homicide_Open_Data.csv',
    'shooting': 'Shootings_Open_Data.csv',
    'hate': 'Hate_Crime_Open_Data.csv'
}

dfs = {}
for name, path in files.items():
    dfs[name] = pd.read_csv(f'./data/raw/{path}')
    print(f"{name}: {dfs[name].shape}")

df_auto, df_bike, df_calls, df_crimes, df_homicide, df_shooting, df_hate = dfs.values()
df_neighbourhoods = gpd.read_file(f'./data/raw/Ottawa_Neighbourhood_Study_(ONS)_-_Neighbourhood_Boundaries_Gen_2.geojson')

print('Dataset loaded and stored in variables.')


auto: (11720, 24)
bike: (15676, 28)
calls: (1108780, 11)
crimes: (344290, 19)
homicide: (131, 13)
shooting: (505, 19)
hate: (1574, 16)
Dataset loaded and stored in variables.


#### Inspect the data
Check the year ranges for consistency

In [3]:
year_cols = {
    'auto':     'Year',
    'bike':     'Year',
    'crimes':   'Year',
    'homicide': 'Year',
    'hate': 'Year',
    'shooting': 'Occurred Year',
    'calls':    'Year',
}

for name, col in year_cols.items():
    years = dfs[name][col]
    print(f"{name}: {years.min()} - {years.max()}")

auto: 2018 - 2025
bike: 2018 - 2025
crimes: 2018 - 2025
homicide: 2018 - 2025
hate: 2021 - 2025
shooting: 2018 - 2025
calls: 2021 - 2025


From the data range we have, we can filter and perform analysis and model trainging for 2018-2025.

### Clean and aggregate datasets by neighbourhood
Each step will perform a count of each record grouped by the neighbourhood.

#### Auto Theft
**Auto Theft → `agg_auto`**: count per neighbourhood; out-of-jurisdiction records dropped.

In [4]:
df_auto.head(1)

,OBJECTID,Vehicle Year,Vehicle Make,Vehicle Model,Vehicle Style,Vehicle Colour,Vehicle Value,Weekday,Recovered,Neighbourhood,Ward,Sector,Reported Date,Occurred Date,Year,Intersection,Division,Census Tract,Time of Day,Councillor,Reported Hour,Occurred Hour,x,y
0,1,2000.0,"HONDA/AMERICAN HONDA MOTOR CO.,",CIVIC (AND CRX),Automobile,DGR,NaN,Monday,Y,Carson Grove - Carson Meadows,Ward 11 - Beacon Hill-Cyrville,Sector 32,1/1/2018 5:00:00 AM,1/1/2018 5:00:00 AM,2018,"CARVER PL, CARVER PL",East,5050122.03,Night,Tim Tierney,700,0,372982.411,5032956.894


In [5]:
# aggreate the number of theft by neighbourhood
# handle out of jurisdiction - drop
agg_auto = df_auto[df_auto['Neighbourhood'] != '<Data Quality - Out of Jurisdiction>'] \
    .groupby('Neighbourhood').agg({'OBJECTID': 'count'}) \
    .reset_index()

# rename the columns
agg_auto.rename(columns={'Neighbourhood':'Neighbourhood','OBJECTID': 'Auto Theft'}, inplace=True)

print(f'We had {agg_auto["Auto Theft"].sum()} auto thefts across {agg_auto.shape[0]} neighbourhoods.')
agg_auto.sort_values('Auto Theft', ascending=False).head()

We had 11189 auto thefts across 111 neighbourhoods.


,Neighbourhood,Auto Theft
72,New Barrhaven - New Development - Stonebridge,605
74,Orléans Avalon - Notting Gate - Fallingbrook -...,587
22,Centretown,413
93,Riverside South - Leitrim,366
33,East Industrial,313


#### Bike Theft
**Bike Theft → `agg_bike`**: count per neighbourhood.

In [6]:
df_bike.head(1)

,OBJECTID,ID,Year,Reported Date,Occurred Date,Day of Week,Offence Category,Bicycle Style,Bicycle Value,Bicycle Make,Bicycle Model,Bicycle Type,Bicycle Frame Size,Bicycle Colour,Bicycle Speed,Neighbourhood,Sector,Division,Census Tract,Status,Intersection,Time of Day,Ward,Councillor,Reported Hour,Occurred Hour,x,y
0,1,1,2018,1/5/2018 5:00:00 AM,1/4/2018 5:00:00 AM,Friday,Theft< Bicycle,Men's,800.0,DIVINCI,SILVERSTONE,Racer,NaN,RED,1.0,Centretown,Sector 23,Central,5050037.02,Stolen,"MACLAREN ST, METCALFE ST",Evening,Ward 14 - Somerset,Ariel Troster,1100,1900,368056.8695,5.031032e+06


In [7]:
# aggreate the number of bike theft by neighbourhood
agg_bike = df_bike.groupby('Neighbourhood').agg({'OBJECTID': 'count'}) \
    .reset_index()

# rename the columns
agg_bike.rename(columns={'Neighbourhood':'Neighbourhood','OBJECTID': 'Bike Theft'}, inplace=True)

print(f'We had {agg_bike["Bike Theft"].sum()} bike thefts across {agg_bike.shape[0]} neighbourhoods.')
agg_bike.sort_values('Bike Theft', ascending=False).head()

We had 15676 bike thefts across 107 neighbourhoods.


,Neighbourhood,Bike Theft
22,Centretown,2437
33,East Industrial,1940
94,Sandy Hill - Ottawa East,1132
13,Byward Market,798
37,Glebe - Dows Lake,728


#### Criminal Offences
**Criminal Offences → `agg_crime`**: count per neighbourhood × offence category, pivoted wide (18 offence types). `Theft - Motor Vehicle` dropped — it duplicates the dedicated auto theft dataset. `Carp` and `Carp Ridge` consolidated into one row — same geographic area, inconsistent naming across datasets.

In [8]:
print(f'Number of criminal categories: {df_crimes['Offence Category'].nunique()}')
df_crimes.head(1)

Number of criminal categories: 18


,OBJECTID,Year,Reported Date,Reported Hour,Occurred Date,Occurred Hour,Weekday,Offence Summary,Offence Category,Neighbourhood Name,Sector,Division,Census Tract,Time of Day,Ward,Councillor,Intersection,x,y
0,1,2018,1/24/2018 5:00:00 AM,1400,1/24/2018 5:00:00 AM,1400,Wednesday,Crimes Against Property (2000),Theft $5000 and Under,Ledbury - Heron Gate - Ridgemont - Elmwood,Sector 34,East,5050007.02,Afternoon,Ward 18 - Alta Vista,Marty Carr,"WALKLEY RD, HEATHERINGTON RD",371766.7279,5.026795e+06


In [9]:
# aggreate the number of crime by neighbourhood and offence category
total_crime = df_crimes.groupby(['Neighbourhood Name', 'Offence Category']) \
    .agg({'OBJECTID': 'count'}) \
    .rename(columns={'OBJECTID': 'Count'}) \
    .reset_index()

# rename the columns
total_crime.rename(columns={'Neighbourhood Name':'Neighbourhood','OBJECTID': 'Count'}, inplace=True)

# pivot the data to have offence categories as columns and fill any missing values with 0
agg_crime = total_crime.pivot(
    index='Neighbourhood',
    columns='Offence Category',
    values='Count'
).fillna(0).astype(int).reset_index().rename_axis(None, axis=1)

print(f'We had {total_crime["Count"].sum()} crimes across {agg_crime.shape[0]} neighbourhoods.')

agg_crime.head()

We had 344104 crimes across 114 neighbourhoods.


,Neighbourhood,Arson,Assaults,Attempting The Commission Of A Capital Crime,Break and Enter,Commodification Of Sexual Activity,Fraud,Mischief,Offensive Weapons,Other Criminal Code,Other Violations Involving Violence Or The Threat Of Violence,Possession / Trafficking Stolen Goods,Prostitution,Sexual Violations,Theft $5000 and Under,Theft - Motor Vehicle,Theft Over $5000,Violations Causing Death,Violations Resulting In The Deprivation Of Freedom
0,Barrhaven,3,208,0,79,0,463,258,10,87,171,0,0,59,346,114,9,3,4
1,Bayshore,3,278,1,129,0,500,244,6,152,203,11,0,73,1722,117,42,0,5
2,Beacon Hill South - Cardinal Heights,5,266,1,107,0,409,210,9,137,187,4,0,76,598,83,23,0,9
3,Beaverbrook,1,87,1,34,0,118,97,0,53,85,0,0,27,113,22,10,0,1
4,Beechwood Cemetery,1,1,0,1,0,3,3,0,2,0,1,0,0,4,0,0,0,0


Notice we have a 'Theft-Motor Vehicle' feature which is similar to the auto theft feature in our initial dataset, so we can drop this.

In [10]:
agg_crime.drop(columns = ['Theft - Motor Vehicle'], inplace=True)

#### Hate Crime
**Hate Crimes → `agg_hate`**: count per neighbourhood × motivation type, pivoted wide (11 categories). Four ambiguous/indeterminate categories (`Combination`, `Not Applicable`, `Other Similar Factor`, `Unknown`) collapsed into a single `Other Hate Crimes` column.

In [11]:
print(f'Number of hate categories: {df_hate['Hate Crime Type'].nunique()}')

df_hate.head(1)

Number of hate categories: 10


,OBJECTID,ID,Year,Reported Date,Occurred Date,Weekday,Hate Crime Type,Hate Crime Motivation,Offence Category,Neighbourhood,Sector,Division,Census Tract,Ward,Councillor,Offence Type
0,1,1,2021,1/7/2021 5:00:00 AM,1/5/2021 5:00:00 AM,Tuesday,Race/Ethnicity,Black,Mischief To Property,Hunt Club Upper -Blossom Park - Timbermill,Sector 35,East,5050123.04,Ward 10 - Gloucester-Southgate,Jessica Bradley,Criminal


In [12]:
total_hate = df_hate.groupby(['Neighbourhood', 'Hate Crime Type']).agg({'OBJECTID': 'count'}) \
    .reset_index()

# rename the columns
total_hate.rename(columns={'OBJECTID': 'Count'}, inplace=True)

agg_hate = total_hate.pivot(
    index='Neighbourhood',
    columns='Hate Crime Type',
    values='Count'
).fillna(0).astype(int).reset_index().rename_axis(None, axis=1)

print(f'We had {total_hate["Count"].sum()} hate crimes across {agg_hate.shape[0]} neighbourhoods.')

agg_hate.head(1)

We had 1573 hate crimes across 104 neighbourhoods.


,Neighbourhood,Combination (More than 2 Motivations),Gender,Immigrants/Newcomers to Canada,Language,Mental or Physical Disability,Other Similar Factor,Race/Ethnicity,Religion,Sexual Orientation,Unknown
0,Barrhaven,0,0,0,0,0,0,4,7,2,0


Collapse ambiguous/indeterminate categories into 'Other Hate Crimes'

In [13]:
ambiguous = [
    'Combination (More than 2 Motivations)',
    'Other Similar Factor',
    'Unknown',
]
agg_hate['Other Hate Crimes'] = agg_hate[ambiguous].sum(axis=1)
agg_hate = agg_hate.drop(columns=ambiguous)
agg_hate.head(1)

,Neighbourhood,Gender,Immigrants/Newcomers to Canada,Language,Mental or Physical Disability,Race/Ethnicity,Religion,Sexual Orientation,Other Hate Crimes
0,Barrhaven,0,0,0,0,4,7,2,0


#### Homicide
**Homicide → `agg_homicide`**: count per neighbourhood.

In [14]:
df_homicide['Offence Category'].value_counts()

Offence Category
Murder 1st Dgree    77
Murder 2nd Dgree    47
Manslaughter         7
Name: count, dtype: int64

In [15]:
df_homicide.head(1)

,OBJECTID,Year,Reported Date,Occurred Date,Weekday,Offence Category,Sector,Division,Neighbourhood,Ward,Councillor,x,y
0,1,2018,1/9/2018 5:00:00 AM,1/9/2018 5:00:00 AM,Wednesday,Murder 1st Dgree,35,East,Hunt Club East - Western Community,Ward 16 - River,Riley Brockington,369457.7211,5.023576e+06


In [16]:
total_homicide = df_homicide.groupby(['Neighbourhood','Offence Category']).agg({'OBJECTID': 'count'}) \
    .reset_index()

# rename the columns
total_homicide.rename(columns={'OBJECTID': 'Count'}, inplace=True)

agg_homicide = total_homicide.pivot(
    index='Neighbourhood',
    columns='Offence Category',
    values='Count'
).fillna(0).astype(int).reset_index().rename_axis(None, axis=1)

print(f'We had {total_homicide["Count"].sum()} homocides across {total_homicide.shape[0]} neighbourhoods.')
agg_homicide.head(3)


We had 131 homocides across 69 neighbourhoods.


,Neighbourhood,Manslaughter,Murder 1st Dgree,Murder 2nd Dgree
0,Barrhaven,0,3,0
1,Bells Corners West,0,0,1
2,Billings Bridge - Alta Vista,0,0,1


#### Shootings
 **Shootings → `agg_shootings`**: count per neighbourhood.

In [17]:
df_shooting.head(1)

,OBJECTID,ID,Reported Date,Reported Hour,Reported Year,Occurred Date,Occurred Hour,Occurred Year,Time of Day,Weekday,Day of Week,Neighbourhood,Sector,Division,Ward,Councillor,Census Tract,x,y
0,1,1,1/3/2018 5:00:00 AM,2200,2018,1/3/2018 5:00:00 AM,2200,2018,Evening,Wednesday,3,Elmvale - Eastway - Riverview - Riverview Park...,33,East,Ward 18 - Alta Vista,Marty Carr,5050008.0,373561.4252,5.027959e+06


In [18]:
agg_shootings = df_shooting.groupby('Neighbourhood').agg({'OBJECTID': 'count'}) \
    .rename(columns={'OBJECTID': 'Shootings'}) \
    .reset_index()

# rename the columns
agg_shootings.rename(columns={'OBJECTID': 'Shootings'}, inplace=True)

print(f'We had {agg_shootings["Shootings"].sum()} shootings across {agg_shootings.shape[0]} neighbourhoods')
agg_shootings.sort_values('Shootings', ascending=False).head()

We had 505 shootings across 94 neighbourhoods


,Neighbourhood,Shootings
73,Overbrook - McArthur,28
54,Ledbury - Heron Gate - Ridgemont - Elmwood,27
12,Byward Market,27
20,Centretown,22
55,Lowertown,21


#### Population
**Population → `df_neighbourhoods`**: neighbourhood boundaries file filtered to `ONS_ID`, `Name`, `POPEST`, `geometry`.

In [19]:
# filter and keeo only necessary columns
df_neighbourhoods = df_neighbourhoods.loc[:,['ONS_ID','Name','POPEST','geometry']]

print(f'We had {df_neighbourhoods["POPEST"].sum()} population across {df_neighbourhoods.shape[0]} neighbourhoods')
df_neighbourhoods.sort_values('POPEST', ascending=False).head()

We had 867146 population across 111 neighbourhoods


,ONS_ID,Name,POPEST,geometry
79,951,Stittsville,26674,"MULTIPOLYGON (((-75.94411 45.28927, -75.94426 ..."
16,24,Centretown,24994,"MULTIPOLYGON (((-75.70993 45.42249, -75.70995 ..."
103,937,Old Barrhaven East,22286,"POLYGON ((-75.7191 45.27663, -75.71863 45.2758..."
62,13,Bridlewood - Emerald Meadows,21101,"POLYGON ((-75.83849 45.28295, -75.83848 45.282..."
35,940,Overbrook - McArthur,19599,"POLYGON ((-75.63261 45.43266, -75.63203 45.432..."


#### Dispatched Calls for Service
**Calls for Service → `agg_police`**: filtered to priority levels 1–3 (dispatched responses only; priorities 4–7 are non-deployed). Mapped to severity labels (`Police_Critical` = life-threatening, `Police_High` = serious harm possible, `Police_Medium` = risk with delay). Counted per neighbourhood × severity, pivoted wide, then joined with `df_neighbourhoods` to attach geometry and population.

We shall aggregate by call priority. Here are the equivalent provided by the OPS
- 1: life-threatening
- 2: serious harm possible
- 3: risk with delay
- 4: mobile response
- 5: broadcast only
- 6: alternate response
- 7: property queue

In [20]:
print(f'Number of calls priorities: {df_calls['Priority'].nunique()}')

df_calls.head(1)

Number of calls priorities: 7


,OBJECTID,Priority,Initiated By,Year,Received Date,Neighbourhood,Time of Day,Received Hour,Day of Week,x,y
0,1,4,C,2021,1/1/2021 5:00:00 AM,908.0,N,0,5,-8425724.984,5.689556e+06


We are just going to keep 3 [1,2,3] priority levels - as these are when a service(s) is depolyed out to the neighbourhood.

In [49]:
# drop any priority other than 1,2,3
df_calls = df_calls[df_calls['Priority'].isin([1, 2, 3])]

# map out the priority levels of police calls
priority_severity = {
    1: 'Police_Critical', # life-threatening
    2: 'Police_High', # serious harm possible
    3: 'Police_Medium', # risk with delay
}
df_calls['Priority Severity'] = df_calls['Priority'].map(priority_severity)

# aggregate data by neighbourhood and priority severity
total_police = df_calls.groupby(['Neighbourhood', 'Priority Severity']).agg({'OBJECTID': 'count'}) \
    .rename(columns={'OBJECTID': 'Count'}) \
    .reset_index()

# pivot the data
agg_police = total_police.pivot(
    index='Neighbourhood',
    columns='Priority Severity',
    values='Count'
).fillna(0).astype(int).reset_index().rename_axis(None, axis=1)

#agg_police.rename(columns={'Neighbourhood':'NB_ID'}, inplace=True)

print(f'We had calls across {agg_police.shape[0]} neighbourhoods')
agg_police.sort_values('Police_Critical', ascending=False).head(1)

We had calls across 111 neighbourhoods


,Neighbourhood,Police_Critical,Police_High,Police_Medium
11,24.0,1317,18859,28926


We can now join calls with the population data as it has the neighbourhod id

In [50]:
agg_police = agg_police.rename(columns={'Neighbourhood': 'ONS_ID'})
agg_police['ONS_ID'] = agg_police['ONS_ID'].astype(int)
agg_police = agg_police.merge(
    df_neighbourhoods[['ONS_ID', 'Name', 'POPEST', 'geometry']].rename(columns={'Name': 'Neighbourhood'}),
    on='ONS_ID',
    how='left'
)

# rename column
agg_police.rename(columns={'POPEST': 'Population'}, inplace=True)

# rearrange
agg_police = agg_police[['ONS_ID', 'Neighbourhood',  'Population', 'Police_Critical', 'Police_High', 'Police_Medium', 'geometry']]

agg_police.head(1)

,ONS_ID,Neighbourhood,Population,Police_Critical,Police_High,Police_Medium,geometry
0,3,Beacon Hill South - Cardinal Heights,7195,75,1037,2141,"POLYGON ((-75.58543 45.44887, -75.58545 45.448..."


### Merge all datasets into a single dataframe

`agg_crime` (113 neighbourhoods) is used as the left base. All other aggregated crime datasets are left-joined on `Neighbourhood` and missing values filled with 0:

```
agg_crime ← agg_auto ← agg_bike ← agg_hate ← agg_homicide ← agg_shootings
```

Twelve columns with ambiguous or low-signal content are summed into a single `Other` feature and the originals are dropped, reducing the merged dataframe from 30 to 19 columns:
- *From criminal offences:* `Commodification Of Sexual Activity`, `Fraud`, `Other Criminal Code`, `Prostitution`
- *From hate crimes:* `Gender`, `Immigrants/Newcomers to Canada`, `Language`, `Mental or Physical Disability`, `Race/Ethnicity`, `Religion`, `Sexual Orientation`, `Other Hate Crimes`


the population and call for service data use different column naming convensions, so we will first aggrage the following, handling the naming discrepancies and then join to a single dataframe for clustering.

In [51]:
# agg_crime has the most neighbourhoods (114), use it as the left base
dfs_to_merge = [agg_auto, agg_bike, agg_hate, agg_homicide, agg_shootings]

df_merged = agg_crime.copy()
for df in dfs_to_merge:
    df_merged = df_merged.merge(df, on='Neighbourhood', how='left')

df_merged = df_merged.fillna(0)
print(f'Merged dataset: {df_merged.shape}')
df_merged.head()

Merged dataset: (114, 32)


,Neighbourhood,Arson,Assaults,Attempting The Commission Of A Capital Crime,Break and Enter,Commodification Of Sexual Activity,Fraud,Mischief,Offensive Weapons,Other Criminal Code,Other Violations Involving Violence Or The Threat Of Violence,Possession / Trafficking Stolen Goods,Prostitution,Sexual Violations,Theft $5000 and Under,Theft Over $5000,Violations Causing Death,Violations Resulting In The Deprivation Of Freedom,Auto Theft,Bike Theft,Gender,Immigrants/Newcomers to Canada,Language,Mental or Physical Disability,Race/Ethnicity,Religion,Sexual Orientation,Other Hate Crimes,Manslaughter,Murder 1st Dgree,Murder 2nd Dgree,Shootings
0,Barrhaven,3,208,0,79,0,463,258,10,87,171,0,0,59,346,9,3,4,103.0,61.0,0.0,0.0,0.0,0.0,4.0,7.0,2.0,0.0,0.0,3.0,0.0,2.0
1,Bayshore,3,278,1,129,0,500,244,6,152,203,11,0,73,1722,42,0,5,119.0,90.0,1.0,2.0,0.0,0.0,5.0,3.0,3.0,1.0,0.0,0.0,0.0,8.0
2,Beacon Hill South - Cardinal Heights,5,266,1,107,0,409,210,9,137,187,4,0,76,598,23,0,9,76.0,60.0,0.0,1.0,0.0,0.0,7.0,6.0,5.0,0.0,0.0,0.0,0.0,3.0
3,Beaverbrook,1,87,1,34,0,118,97,0,53,85,0,0,27,113,10,0,1,23.0,33.0,0.0,0.0,0.0,0.0,3.0,6.0,1.0,0.0,0.0,0.0,0.0,0.0
4,Beechwood Cemetery,1,1,0,1,0,3,3,0,2,0,1,0,0,4,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


We can combine the following petty crime features into a 'Other' feature based on their nature.

In [52]:
other_cols = [
    'Commodification Of Sexual Activity', 'Fraud', 'Other Criminal Code', 'Prostitution',
    'Gender', 'Immigrants/Newcomers to Canada', 'Language', 'Mental or Physical Disability',
    'Race/Ethnicity', 'Religion', 'Sexual Orientation', 'Other Hate Crimes',
]
df_merged['Other'] = df_merged[other_cols].sum(axis=1)
df_merged = df_merged.drop(columns=other_cols)
print(f'df_merged shape after combining: {df_merged.shape}')
df_merged.head(2)

df_merged shape after combining: (114, 21)


,Neighbourhood,Arson,Assaults,Attempting The Commission Of A Capital Crime,Break and Enter,Mischief,Offensive Weapons,Other Violations Involving Violence Or The Threat Of Violence,Possession / Trafficking Stolen Goods,Sexual Violations,Theft $5000 and Under,Theft Over $5000,Violations Causing Death,Violations Resulting In The Deprivation Of Freedom,Auto Theft,Bike Theft,Manslaughter,Murder 1st Dgree,Murder 2nd Dgree,Shootings,Other
0,Barrhaven,3,208,0,79,258,10,171,0,59,346,9,3,4,103.0,61.0,0.0,3.0,0.0,2.0,563.0
1,Bayshore,3,278,1,129,244,6,203,11,73,1722,42,0,5,119.0,90.0,0.0,0.0,0.0,8.0,667.0


### String matching

The crime dataset and the calls-for-service dataset use different neighbourhood naming conventions (69 of 113 match exactly out of the box). Mismatches are resolved in three passes:
1. Polygon Unions
2. Manual overrides
3. Fuzzy matching

Step 1: Find mismatched neigbourhood names across both datasets

In [53]:
# preview mismatches
map_names = set(agg_police['Neighbourhood'])
df_names = set(df_merged['Neighbourhood'])

print(f"Matched: {len(map_names & df_names)}")
print(f"\nIn map but not in df:\n{sorted(map_names - df_names)}")
print(f"\nIn df but not in map:\n{sorted(df_names - map_names)}")

Matched: 69

In map but not in df:
['Bayshore - Belltown', 'Borden Farm - Fisher Glen', "Brookside - Briarbrook - Morgan's Grant", 'Cardinal Creek', 'Chapel Hill North', 'Chapel Hill South', 'Chapman Mills', 'Chatelaine Village', 'Cityview - Crestview - Meadowlands', 'Convent Glen - Orléans Woods', 'Edwards - Carlsbad Springs', 'Elmvale - Canterbury', 'Fallingbrook', 'Findlay Creek', 'Greenbelt', 'Iris - Queensway Terrance South', 'Island Park - Wellington Village', 'Kanata Lakes - Arcardia', 'Ledbury - Heron Gate - Ridgemont', 'Manor Park', 'Manotick', 'Marlborough', 'Munster - Ashton', 'Navan - Sarsfield', 'North Gower - Kars', 'Old Barrhaven East', 'Old Barrhaven West', 'Old Ottawa East', 'Old Ottawa South', 'Osgoode - Vernon', 'Parkwood Hills - Stewart Farm', 'Portobello South', 'Queenswood Heights', 'Rideau Crest - Davidson Heights', 'Riverview', 'Rockcliffe Park', 'Sandy Hill', 'Skyline - Fisher Heights', 'South Keys - Greenboro West', "Stonebridge - Half Moon Bay - Heart's Desir

#### 1. Polygon Unions
Several crime-dataset neighbourhood names correspond to multiple map polygons. Their geometries are unioned in `agg_police` before any name matching (e.g. `Barrhaven` ← Old Barrhaven West + Old Barrhaven East; `Orléans Avalon…` ← Fallingbrook + Portobello South + Cardinal Creek).

In [55]:
# df neighbourhood -> constituent map neighbourhoods whose geometries should be unioned
combinations = {
    'Barrhaven':                                                 ['Old Barrhaven West', 'Old Barrhaven East'],
    'New Barrhaven - New Development - Stonebridge':             ["Stonebridge - Half Moon Bay - Heart's Desire"],
    'Rockcliffe - Manor Park':                                   ['Rockcliffe Park', 'Manor Park'],
    'Chapman Mills - Rideau Crest - Davidson Heights':           ['Chapman Mills', 'Rideau Crest - Davidson Heights'],
    'Cityview - Skyline - Fisher Heights':                       ['Cityview - Crestview - Meadowlands', 'Skyline - Fisher Heights'],
    'Borden Farm - Stewart Farm - Parkwood Hills - Fisher Glen': ['Parkwood Hills - Stewart Farm', 'Borden Farm - Fisher Glen'],
    'Sandy Hill - Ottawa East':                                  ['Sandy Hill', 'Old Ottawa East'],
    'Riverside South - Leitrim':                                 ['Riverside South - Leitrim', 'Findlay Creek'],
    "Kanata Lakes - Marchwood Lakeside - Morgan's Grant - Kanata North Business Park": ['Kanata Lakes - Arcardia', "Brookside - Briarbrook - Morgan's Grant"],
    'Elmvale - Eastway - Riverview - Riverview Park West':       ['Riverview', 'Elmvale - Canterbury'],
    'Orléans Avalon - Notting Gate - Fallingbrook - Gardenway South': ['Fallingbrook', 'Portobello South', 'Cardinal Creek'],
}

# build all combined rows first (parts stay in agg_police during iteration),
# then remove all consumed parts in one pass — allows multiple keys to share the same parts
all_parts_to_remove = set()
combined_rows = []

for df_name, parts in combinations.items():
    subset = agg_police[agg_police['Neighbourhood'].isin(parts)]
    missing = set(parts) - set(subset['Neighbourhood'])
    if missing:
        print(f"Warning: parts not found for '{df_name}': {missing}")
        continue
    combined_rows.append(gpd.GeoDataFrame([{
        'Neighbourhood':   df_name,
        'Population':      int(subset['Population'].sum()),
        'Police_Critical': int(subset['Police_Critical'].sum()),
        'Police_High':     int(subset['Police_High'].sum()),
        'Police_Medium':   int(subset['Police_Medium'].sum()),
        'geometry':        gpd.GeoSeries(subset['geometry']).union_all(),
    }], crs=df_neighbourhoods.crs))
    all_parts_to_remove.update(parts)

agg_police = agg_police[~agg_police['Neighbourhood'].isin(all_parts_to_remove)]
agg_police = pd.concat([agg_police] + combined_rows, ignore_index=True)

print(f"agg_police shape: {agg_police.shape}")

agg_police shape: (100, 7)


2. **Manual overrides**: A lookup table of confirmed low-confidence or ambiguous matches is applied first (e.g. `Bayshore` → `Bayshore - Belltown`, `CFB Rockcliffe-NRC` → `Wateridge Village`, `Ottawa East` → `Sandy Hill - Ottawa East`).
3. **Fuzzy matching**: Remaining names matched using `rapidfuzz` token-sort ratio at a threshold of 70. Result: 104 of 113 names successfully mapped.

In [56]:
# set the threshold 
THRESHOLD = 70

map_name_list = agg_police['Neighbourhood'].tolist()
df_name_list = df_merged['Neighbourhood'].tolist()

# manual overrides for confirmed low-confidence or ambiguous matches
manual_overrides = {
    'Bayshore':                                                              'Bayshore - Belltown',
    'Osgoode':                                                               'Osgoode - Vernon',
    'Sarsfield':                                                             'Navan - Sarsfield',
    'Russell - Edwards':                                                     'Edwards - Carlsbad Springs',
    'Munster':                                                               'Munster - Ashton',
    'CFB Rockcliffe-NRC':                                                    'Wateridge Village',
    'Island Park':                                                           'Island Park - Wellington Village',
    'Iris':                                                                  'Iris - Queensway Terrance South',
    'Orléans North West':  'Convent Glen - Orléans Woods',                                                  
    'Orléans Industrial':'Orléans Industrial',
    'Orléans Village - Chateauneuf':        'Orléans Village - Chateauneuf',
    'Orléans Chatelaine Village' :'Chatelaine Village',
    
    'Pierces Corners':'Marlborough',
    
    # Manotick split in crimes but single zone in calls
    'Manotick East':                                                         'Manotick',
    'Manotick West':                                                         'Manotick',
    'North Gower':                                                           'North Gower - Kars',
    # Ottawa East covers Old Ottawa East, which was merged into Sandy Hill - Ottawa East
    'Ottawa East':                                                           'Sandy Hill - Ottawa East',
    # Crestview - Meadowlands is part of the combined Cityview - Skyline zone
    'Crestview - Meadowlands':                                               'Cityview - Skyline - Fisher Heights',
}

fuzzy_map = {}
unmatched = []

for name in df_name_list:
    if name in manual_overrides:
        fuzzy_map[name] = manual_overrides[name]
        continue
    result = process.extractOne(name, map_name_list, scorer=fuzz.token_sort_ratio)
    if result and result[1] >= THRESHOLD:
        fuzzy_map[name] = result[0]
    else:
        unmatched.append((name, result))

print(f"Mapped:          {len(fuzzy_map)} / {len(df_name_list)}")
print(f"Still unmatched: {len(unmatched)}")
if unmatched:
    print("\nUnmatched (review manually):")
    for name, result in unmatched:
        print(f"  '{name}' -> best: '{result[0]}' ({result[1]:.0f})")

Mapped:          104 / 114
Still unmatched: 10

Unmatched (review manually):
  'Carp Ridge' -> best: 'Riverside Park' (58)
  'Cummings' -> best: 'Carlington' (44)
  'Galetta' -> best: 'Laurentian' (47)
  'Greenbelt - Mer Bleue' -> best: 'Greenbelt' (60)
  'Greenbelt - Rideau River East' -> best: 'Carleton Heights - Rideauview' (59)
  'Greenbelt - Rideau River West' -> best: 'South Keys - Greenboro West' (57)
  'Greenbelt - Shirleys Bay' -> best: 'Briar Green - Leslie Park' (57)
  'Greenbelt - Stony Swamp' -> best: 'Greenbelt' (56)
  'Greenbelt SouthEast' -> best: 'South Keys - Greenboro West' (65)
  'Woodroffe - Lincoln Heights' -> best: 'Rothwell Heights - Beacon Hill North' (54)


## 2. Join both datasets

The crime dataset is inner-joined with `agg_police` on the resolved keys, attaching population, geometry, and police call counts to each row.

In [57]:
# keep only the mapped neighbourhoods and attach their agg_police equivalent name
df_mapped = df_merged[df_merged['Neighbourhood'].isin(fuzzy_map)].copy()
df_mapped['police_key'] = df_mapped['Neighbourhood'].map(fuzzy_map)

# join with agg_police on the mapped key; Neighbourhood stays as the crime-dataset name
df_final = df_mapped.merge(
    agg_police.rename(columns={'Neighbourhood': 'police_key'}),
    on='police_key',
    how='inner'
).drop(columns='police_key')

print(f'Final dataset: {df_final.shape}')

Final dataset: (104, 27)


Nine Greenbelt sub-areas (e.g. `Greenbelt - Mer Bleue`, `Greenbelt SouthEast`) are excluded — their boundaries are ambiguous across datasets and they contain no residential population relevant to the safety analysis. The remaining 9 unmatched names are rural/edge-case areas with no counterpart in the calls-for-service data.

In [61]:
# drop Greenbelt neighbourhoods — boundaries are ambiguous/indecisive
df_final = df_final[~df_final['Neighbourhood'].str.contains('Greenbelt', case=False, na=False)]
df_final.drop(columns=['ONS_ID'], inplace=True)
print(f'df_final shape after dropping Greenbelt: {df_final.shape}')

df_final shape after dropping Greenbelt: (104, 26)


In [62]:
df_final.head(2)

,Neighbourhood,Arson,Assaults,Attempting The Commission Of A Capital Crime,Break and Enter,Mischief,Offensive Weapons,Other Violations Involving Violence Or The Threat Of Violence,Possession / Trafficking Stolen Goods,Sexual Violations,Theft $5000 and Under,Theft Over $5000,Violations Causing Death,Violations Resulting In The Deprivation Of Freedom,Auto Theft,Bike Theft,Manslaughter,Murder 1st Dgree,Murder 2nd Dgree,Shootings,Other,Population,Police_Critical,Police_High,Police_Medium,geometry
0,Barrhaven,3,208,0,79,258,10,171,0,59,346,9,3,4,103.0,61.0,0.0,3.0,0.0,2.0,563.0,39166,185,2592,5100,"POLYGON ((-75.77423 45.26102, -75.77576 45.262..."
1,Bayshore,3,278,1,129,244,6,203,11,73,1722,42,0,5,119.0,90.0,0.0,0.0,0.0,8.0,667.0,9313,80,1089,2501,"POLYGON ((-75.80236 45.35727, -75.80236 45.357..."


#### Export data for model building and cluster analysis

In [ ]:
df_final.to_csv('CleanedData.csv', index=False)
print('Cleaned data saved.')